In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

# --- Signal Parameters for Frequency-Domain Textbook Replication ---
Omega_N = 3000.0 * 2 * np.pi  # Maximum frequency of the continuous signal in rad/s
F_N = Omega_N / (2 * np.pi)

@widgets.interact(Fs_ratio=widgets.FloatSlider(value=2.5, min=1.5, max=4.0, step=0.1, description='Ratio $F_s / F_{max}$:', style={'description_width': 'initial'}, layout=widgets.Layout(width='700px')))
def update_frequency_textbook_plot(Fs_ratio):
    Fs = Fs_ratio * F_N
    Omega_s = 2 * np.pi * Fs
    Ts = 1.0 / Fs
    
    # Frequency axis in rad/s
    omega_max = 5 * Omega_N
    omega = np.linspace(-omega_max, omega_max, 3000)
    
    fig = plt.figure(figsize=(12, 10))
    gs = fig.add_gridspec(3, 2, hspace=0.4, wspace=0.3)
    
    # --- (a) Spectrum of continuous time signal X_a(jΩ) ---
    ax_a = fig.add_subplot(gs[0, 0])
    ax_a.grid(True, linestyle=':', alpha=0.7)
    ax_a.set_xlim(-2 * Omega_N, 2 * Omega_N)
    ax_a.set_ylim(-0.2, 1.3)
    ax_a.set_ylabel(r'$X_a(j\Omega)$', fontsize=10)
    ax_a.set_title(r'(a) Spectrum of continuous signal', fontsize=10, fontweight='bold')
    
    # Triangular spectrum function
    X_a = np.where(np.abs(omega) <= Omega_N, 1 - np.abs(omega) / Omega_N, 0)
    ax_a.plot(omega, X_a, color='red', linewidth=1.5)
    ax_a.fill_between(omega, X_a, color='red', alpha=0.2)
    ax_a.set_xticks([-Omega_N, 0, Omega_N])
    ax_a.set_xticklabels([r'$-\Omega_N$', '0', r'$\Omega_N$'])
    ax_a.set_yticks([1])
    ax_a.set_yticklabels(['1'])
    
    # --- (d) Ideal low pass filter H_r(jΩ) ---
    ax_d = fig.add_subplot(gs[0, 1])
    ax_d.grid(True, linestyle=':', alpha=0.7)
    ax_d.set_xlim(-2 * Omega_N, 2 * Omega_N)
    ax_d.set_ylim(-0.2, Ts * 1.5)
    ax_d.set_ylabel(r'$H_r(j\Omega)$', fontsize=10)
    ax_d.set_title(r'(d) Ideal lowpass filter', fontsize=10, fontweight='bold')
    
    Omega_c = Omega_s / 2  # Cutoff frequency
    H_r = np.where(np.abs(omega) <= Omega_c, Ts, 0)
    ax_d.plot(omega, H_r, color='red', linewidth=1.5)
    ax_d.fill_between(omega, H_r, color='red', alpha=0.2)
    ax_d.set_xticks([-Omega_c, 0, Omega_c])
    ax_d.set_xticklabels([r'$-\Omega_c$', '0', r'$\Omega_c$'])
    ax_d.set_yticks([Ts])
    ax_d.set_yticklabels([r'$T_s$'])

    # --- (b) Spectrum of sampling function S(jΩ) ---
    ax_b = fig.add_subplot(gs[1, :])
    ax_b.grid(True, linestyle=':', alpha=0.7)
    ax_b.set_xlim(-3.5 * Omega_s, 3.5 * Omega_s)
    ax_b.set_ylim(-0.5, (2 * np.pi / Ts) * 1.3)
    ax_b.set_ylabel(r'$S(j\Omega)$', fontsize=10)
    ax_b.set_title(r'(b) Spectrum of sampling function $s(t)$', fontsize=10, fontweight='bold')
    
    # Impulses at k * Omega_s
    k_vals = np.arange(-3, 4)
    for k in k_vals:
        loc = k * Omega_s
        ax_b.plot([loc, loc], [0, 2 * np.pi / Ts], color='red', linewidth=1.5)
        ax_b.plot(loc, 2 * np.pi / Ts, marker='^', color='red', markersize=5)
        
    ax_b.set_xticks([k * Omega_s for k in range(-3, 4)])
    ax_b.set_xticklabels([r'$-3\Omega_s$', r'$-2\Omega_s$', r'$-\Omega_s$', '0', r'$\Omega_s$', r'$2\Omega_s$', r'$3\Omega_s$'])
    ax_b.set_yticks([2 * np.pi / Ts])
    ax_b.set_yticklabels([r'$\frac{2\pi}{T_s}$'])

    # --- (c) Spectrum of sampled signal X_s(jΩ) ---
    ax_c = fig.add_subplot(gs[2, :])
    ax_c.grid(True, linestyle=':', alpha=0.7)
    ax_c.set_xlim(-3.5 * Omega_s, 3.5 * Omega_s)
    ax_c.set_ylim(-0.2, (1 / Ts) * 1.4)
    ax_c.set_xlabel(r'Frequency $\Omega$ (rad/s)', fontsize=10)
    ax_c.set_ylabel(r'$X_s(j\Omega)$', fontsize=10)
    ax_c.set_title(r'(c) Spectrum of sampled signal (Periodic replication)', fontsize=10, fontweight='bold')
    
    # Replicas sum
    X_s = np.zeros_like(omega)
    for k in range(-4, 5):
        shift = k * Omega_s
        replica = (1.0 / Ts) * np.where(np.abs(omega - shift) <= Omega_N, 1 - np.abs(omega - shift) / Omega_N, 0)
        X_s += replica
        ax_c.plot(omega, replica, color='red', alpha=0.5, linestyle='--')
        
    ax_c.plot(omega, X_s, color='red', linewidth=1.5)
    ax_c.fill_between(omega, X_s, color='red', alpha=0.2)
    
    ax_c.set_xticks([-2 * Omega_s, -Omega_s, -Omega_N, Omega_N, Omega_s, 2 * Omega_s, 3 * Omega_s])
    ax_c.set_xticklabels([r'$-2\Omega_s$', r'$-\Omega_s$', r'$-\Omega_N$', r'$\Omega_N$', r'$\Omega_s$', r'$2\Omega_s$', r'$3\Omega_s$'])
    ax_c.set_yticks([1 / Ts])
    ax_c.set_yticklabels([r'$\frac{1}{T_s}$'])

    plt.show()

    # --- Educational Explanation ---
    print("\n" + "="*95)
    print(" EDUCATIONAL GUIDE FOR TEXTBOOK-STYLE FREQUENCY REPLICATION PLOTS:")
    print("="*95)
    print(" - (a) Shows the baseband triangular spectrum of the continuous signal bounded by +- Omega_N.")
    print(" - (b) Displays the Dirac impulse train in the frequency domain for the sampling function s(t).")
    print(" - (c) Demonstrates the exact periodic replication where identical upright triangular replicas")
    print("       are placed at integer multiples of Omega_s, scaled by 1/Ts, matching standard textbooks!")
    print(" - (d) Represents the ideal reconstruction low-pass filter with passband Omega_c = Omega_s/2 and height Ts.")
    print("="*95)